In [9]:
import xml.etree.ElementTree as ET
import csv
import os
import glob
from typing import Dict, List, Tuple, Any, Optional

# def parse_arggraph(xml_str: str, filename: Optional[str] = None) -> List[Dict[str, str]]:
#     """Parse ArgGraph XML and extract relations between nodes."""
#     try:
#         # Parse XML content
#         root = ET.fromstring(xml_str)
#     except ET.ParseError as e:
#         print(f"Failed to parse XML: {e}")
#         return []
    
#     # Create mapping from node ID to text
#     node_texts: Dict[str, str] = {}
    
#     # Extract edu nodes text
#     for edu in root.findall('.//edu'):
#         node_id = edu.get('id')
#         if node_id is None:
#             continue
            
#         # Extract text content, handling CDATA
#         text = edu.text.strip() if edu.text else ""
        
#         # Handle CDATA sections
#         if text.startswith('<![CDATA[') and text.endswith(']]>'):
#             text = text[9:-3].strip()
        
#         node_texts[node_id] = text
    
#     # For adu nodes, find corresponding edu text via seg edges
#     for edge in root.findall('.//edge[@type="seg"]'):
#         edu_id = edge.get('src')
#         adu_id = edge.get('trg')
#         if edu_id and adu_id and edu_id in node_texts:
#             node_texts[adu_id] = node_texts[edu_id]
    
#     # Extract all edges and create mapping
#     edge_info: Dict[str, Dict[str, str]] = {}
#     for edge in root.findall('.//edge'):
#         edge_id = edge.get('id')
#         if edge_id is None:
#             continue
            
#         edge_info[edge_id] = {
#             'src': edge.get('src', ''),
#             'trg': edge.get('trg', ''),
#             'type': edge.get('type', '')
#         }
    
#     # Build the final dataset
#     dataset: List[Dict[str, str]] = []
    
#     for edge in root.findall('.//edge'):
#         edge_type = edge.get('type')
#         if edge_type is None:
#             continue
            
#         # Skip seg edges (already handled)
#         if edge_type == 'seg':
#             continue
            
#         edge_id = edge.get('id')
#         src_id = edge.get('src')
#         trg_id = edge.get('trg')
#         if src_id is None or trg_id is None:
#             continue
        
#         # Handle und relations (undercut) - special case
#         if edge_type == 'und':
#             # For und relations, the target is an edge ID, not a node ID
#             if trg_id in edge_info:
#                 target_edge = edge_info[trg_id]
#                 # X is the source of the und edge (the undercutter)
#                 x_text = node_texts.get(src_id, "")
#                 # Y is the source of the target edge (the claim being undercut)
#                 y_text = node_texts.get(target_edge['src'], "")
                
#                 dataset.append({
#                     'X': x_text,
#                     'Y': y_text,
#                     'Z': edge_type,
#                     'Source': filename or 'unknown'
#                 })
#             continue
        
#         # For other relation types (sup, reb, etc.)
#         x_text = node_texts.get(src_id, "")
#         y_text = node_texts.get(trg_id, "")
        
#         dataset.append({
#             'X': x_text,
#             'Y': y_text,
#             'Z': edge_type,
#             'Source': filename or 'unknown'
#         })
    
#     return dataset

def process_xml_files(folder_path: str, separate_files: bool = True) -> List[Dict[str, str]]:
    """Process all XML files in the specified folder."""
    xml_files = glob.glob(os.path.join(folder_path, "*.xml"))
    
    if not xml_files:
        print(f"No XML files found in {folder_path}")
        return []
    
    all_data = []
    
    for xml_file in xml_files:
        print(f"Processing: {xml_file}")
        try:
            with open(xml_file, 'r', encoding='utf-8') as file:
                xml_content = file.read()
                filename = os.path.basename(xml_file)
                data = parse_arggraph(xml_content, filename)
                
                if separate_files:
                    base_name = os.path.splitext(filename)[0]
                    csv_filename = f"{base_name}_dataset.csv"
                    
                    with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
                        fieldnames = ['X', 'Y', 'Z','topic', 'Source']
                        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                        writer.writeheader()
                        writer.writerows(data)
                    
                    print(f"Saved to {csv_filename}")
                
                all_data.extend(data)
                print(f"Extracted {len(data)} relations from {filename}")
                
        except Exception as e:
            print(f"Error processing {xml_file}: {e}")
    
    return all_data
# def parse_arggraph(xml_str: str, filename: Optional[str] = None) -> List[Dict[str, str]]:
#     """Parse ArgGraph XML and extract relations between nodes."""
#     try:
#         # Parse XML content
#         root = ET.fromstring(xml_str)
#     except ET.ParseError as e:
#         print(f"Failed to parse XML: {e}")
#         return []
    
#     # Create mapping from node ID to text
#     node_texts: Dict[str, str] = {}
    
#     # Extract edu nodes text
#     for edu in root.findall('.//edu'):
#         node_id = edu.get('id')
#         if node_id is None:
#             continue
            
#         # Extract text content, handling CDATA
#         text = edu.text.strip() if edu.text else ""
        
#         # Handle CDATA sections
#         if text.startswith('<![CDATA[') and text.endswith(']]>'):
#             text = text[9:-3].strip()
        
#         node_texts[node_id] = text
    
#     # For adu nodes, find corresponding edu text via seg edges
#     for edge in root.findall('.//edge[@type="seg"]'):
#         edu_id = edge.get('src')
#         adu_id = edge.get('trg')
#         if edu_id and adu_id and edu_id in node_texts:
#             node_texts[adu_id] = node_texts[edu_id]
    
#     # Extract all edges and create mapping
#     edge_info: Dict[str, Dict[str, str]] = {}
#     for edge in root.findall('.//edge'):
#         edge_id = edge.get('id')
#         if edge_id is None:
#             continue
            
#         edge_info[edge_id] = {
#             'src': edge.get('src', ''),
#             'trg': edge.get('trg', ''),
#             'type': edge.get('type', '')
#         }
    
#     # Identify edges that will be combined with add relations
#     edges_to_skip = set()
#     for edge in root.findall('.//edge[@type="add"]'):
#         trg_id = edge.get('trg')
#         if trg_id in edge_info:
#             edges_to_skip.add(trg_id)
    
#     # Build the final dataset
#     dataset: List[Dict[str, str]] = []
    
#     for edge in root.findall('.//edge'):
#         edge_type = edge.get('type')
#         if edge_type is None:
#             continue
            
#         # Skip seg edges (already handled)
#         if edge_type == 'seg':
#             continue
            
#         edge_id = edge.get('id')
#         src_id = edge.get('src')
#         trg_id = edge.get('trg')
#         if src_id is None or trg_id is None:
#             continue
        
#         # Skip edges that will be combined with add relations
#         if edge_id in edges_to_skip:
#             continue
        
#         # Handle und relations (undercut) - special case
#         if edge_type == 'und':
#             # For und relations, the target is an edge ID, not a node ID
#             if trg_id in edge_info:
#                 target_edge = edge_info[trg_id]
#                 # X is the source of the und edge (the undercutter)
#                 x_text = node_texts.get(src_id, "")
#                 # Y is the source of the target edge (the claim being undercut)
#                 y_text = node_texts.get(target_edge['src'], "")
                
#                 dataset.append({
#                     'X': x_text,
#                     'Y': y_text,
#                     'Z': edge_type,
#                     'Source': filename or 'unknown'
#                 })
#             continue
        
#         # Handle add relations - special case (combine with target edge's source)
#         if edge_type == 'add':
#             # For add relations, the target is an edge ID
#             if trg_id in edge_info:
#                 target_edge = edge_info[trg_id]
#                 # Get the source of the target edge (a5 in your example)
#                 target_src_id = target_edge['src']
#                 # Get the target of the target edge (a1 in your example)
#                 target_trg_id = target_edge['trg']
                
#                 # Combine texts: X is the add edge source + target edge source
#                 add_text = node_texts.get(src_id, "")
#                 target_src_text = node_texts.get(target_src_id, "")
#                 combined_text = f"{target_src_text} {add_text}".strip()
                
#                 # Y is the target of the target edge
#                 y_text = node_texts.get(target_trg_id, "")
                
#                 dataset.append({
#                     'X': combined_text,
#                     'Y': y_text,
#                     'Z': target_edge['type'],  # Use the type of the target edge
#                     'Source': filename or 'unknown'
#                 })
#             continue
        
#         # For other relation types (sup, reb, etc.)
#         x_text = node_texts.get(src_id, "")
#         y_text = node_texts.get(trg_id, "")
        
#         dataset.append({
#             'X': x_text,
#             'Y': y_text,
#             'Z': edge_type,
#             'Source': filename or 'unknown'
#         })
    
#     return dataset
# def parse_arggraph(xml_str: str, filename: Optional[str] = None) -> List[Dict[str, str]]:
#     """Parse ArgGraph XML and extract relations between nodes."""
#     try:
#         # Parse XML content
#         root = ET.fromstring(xml_str)
#     except ET.ParseError as e:
#         print(f"Failed to parse XML: {e}")
#         return []
    
#     # Create mapping from node ID to text
#     node_texts: Dict[str, str] = {}
    
#     # Extract edu nodes text
#     for edu in root.findall('.//edu'):
#         node_id = edu.get('id')
#         if node_id is None:
#             continue
            
#         # Extract text content, handling CDATA
#         text = edu.text.strip() if edu.text else ""
        
#         # Handle CDATA sections
#         if text.startswith('<![CDATA[') and text.endswith(']]>'):
#             text = text[9:-3].strip()
        
#         node_texts[node_id] = text
    
#     # For adu nodes, find corresponding edu text via seg edges
#     for edge in root.findall('.//edge[@type="seg"]'):
#         edu_id = edge.get('src')
#         adu_id = edge.get('trg')
#         if edu_id and adu_id and edu_id in node_texts:
#             node_texts[adu_id] = node_texts[edu_id]
    
#     # Extract all edges and create mapping
#     edge_info: Dict[str, Dict[str, str]] = {}
#     for edge in root.findall('.//edge'):
#         edge_id = edge.get('id')
#         if edge_id is None:
#             continue
            
#         edge_info[edge_id] = {
#             'src': edge.get('src', ''),
#             'trg': edge.get('trg', ''),
#             'type': edge.get('type', '')
#         }
    
#     # Build the final dataset
#     dataset: List[Dict[str, str]] = []
    
#     # First process all non-special edges
#     for edge in root.findall('.//edge'):
#         edge_type = edge.get('type')
#         if edge_type is None or edge_type in ['seg', 'und', 'add']:
#             continue
            
#         src_id = edge.get('src')
#         trg_id = edge.get('trg')
#         if src_id is None or trg_id is None:
#             continue
        
#         x_text = node_texts.get(src_id, "")
#         y_text = node_texts.get(trg_id, "")
        
#         dataset.append({
#             'X': x_text,
#             'Y': y_text,
#             'Z': edge_type,
#             'Source': filename or 'unknown'
#         })
    
#     # Process special edges (und, add) after regular ones
#     for edge in root.findall('.//edge'):
#         edge_type = edge.get('type')
#         if edge_type is None:
#             continue
            
#         # Handle und relations (undercut)
#         if edge_type == 'und':
#             src_id = edge.get('src')
#             trg_id = edge.get('trg')
#             if src_id is None or trg_id is None:
#                 continue
                
#             # For und relations, the target is an edge ID
#             if trg_id in edge_info:
#                 target_edge = edge_info[trg_id]
#                 # X is the source of the und edge (the undercutter)
#                 x_text = node_texts.get(src_id, "")
#                 # Y is the source of the target edge (the claim being undercut)
#                 y_text = node_texts.get(target_edge['src'], "")
                
#                 dataset.append({
#                     'X': x_text,
#                     'Y': y_text,
#                     'Z': edge_type,
#                     'Source': filename or 'unknown'
#                 })
        
#         # Handle add relations
#         elif edge_type == 'add':
#             src_id = edge.get('src')
#             trg_id = edge.get('trg')
#             if src_id is None or trg_id is None:
#                 continue
                
#             # For add relations, the target is an edge ID
#             if trg_id in edge_info:
#                 target_edge = edge_info[trg_id]
#                 # Check if the target edge is an undercut
#                 if target_edge['type'] == 'und':
#                     # Get the und edge's target (which is another edge)
#                     und_target_id = target_edge['trg']
#                     if und_target_id in edge_info:
#                         und_target_edge = edge_info[und_target_id]
#                         # X is the combined text of add source + und source
#                         add_text = node_texts.get(src_id, "")
#                         und_text = node_texts.get(target_edge['src'], "")
#                         combined_text = f"{und_text} {add_text}".strip()
#                         # Y is the source of the edge being undercut
#                         y_text = node_texts.get(und_target_edge['src'], "")
                        
#                         dataset.append({
#                             'X': combined_text,
#                             'Y': y_text,
#                             'Z': 'und',  # This is still an undercut relation
#                             'Source': filename or 'unknown'
#                         })
    
#     return dataset
def parse_arggraph(xml_str: str, filename: Optional[str] = None) -> List[Dict[str, str]]:
    """Parse ArgGraph XML and extract relations between nodes."""
    try:
        # Parse XML content
        root = ET.fromstring(xml_str)
    except ET.ParseError as e:
        print(f"Failed to parse XML: {e}")
        return []
    topic_id = root.get('topic_id', 'unknown_topic')
    print(topic_id)
    # Create mapping from node ID to text
    node_texts: Dict[str, str] = {}
    
    # Extract edu nodes text
    for edu in root.findall('.//edu'):
        node_id = edu.get('id')
        if node_id is None:
            continue
            
        # Extract text content, handling CDATA
        text = edu.text.strip() if edu.text else ""
        
        # Handle CDATA sections
        if text.startswith('<![CDATA[') and text.endswith(']]>'):
            text = text[9:-3].strip()
        
        node_texts[node_id] = text
    
    # For adu nodes, find corresponding edu text via seg edges
    for edge in root.findall('.//edge[@type="seg"]'):
        edu_id = edge.get('src')
        adu_id = edge.get('trg')
        if edu_id and adu_id and edu_id in node_texts:
            node_texts[adu_id] = node_texts[edu_id]
    
    # Extract all edges and create mapping
    edge_info: Dict[str, Dict[str, str]] = {}
    for edge in root.findall('.//edge'):
        edge_id = edge.get('id')
        if edge_id is None:
            continue
            
        edge_info[edge_id] = {
            'src': edge.get('src', ''),
            'trg': edge.get('trg', ''),
            'type': edge.get('type', '')
        }
    
    # Track which edges have been processed as part of add relations
    processed_edges = set()
    
    # Build the final dataset
    dataset: List[Dict[str, str]] = []
    
    # First process add relations that target undercut edges
    for edge in root.findall('.//edge[@type="add"]'):
        edge_id = edge.get('id')
        src_id = edge.get('src')
        trg_id = edge.get('trg')
        
        if not all([edge_id, src_id, trg_id]):
            continue
            
        # Check if this add targets an undercut edge
        if trg_id in edge_info and edge_info[trg_id]['type'] == 'und':
            und_edge = edge_info[trg_id]
            und_src_id = und_edge['src']
            und_trg_id = und_edge['trg']
            
            # Get the edge that the undercut targets
            if und_trg_id in edge_info:
                target_edge = edge_info[und_trg_id]
                target_src_id = target_edge['src']
                
                # Combine the texts
                add_text = node_texts.get(src_id, "")
                und_text = node_texts.get(und_src_id, "")
                combined_text = f"{und_text} {add_text}".strip()
                target_text = node_texts.get(target_src_id, "")
                
                dataset.append({
                    'X': combined_text,
                    'Y': target_text,
                    'Z': 'und',
                    'topic': topic_id,
                    'Source': filename or 'unknown'
                })
                
                # Mark both edges as processed
                processed_edges.add(edge_id)
                processed_edges.add(trg_id)
    
    # Then process all other edges
    for edge in root.findall('.//edge'):
        edge_id = edge.get('id')
        edge_type = edge.get('type')
        src_id = edge.get('src')
        trg_id = edge.get('trg')
        
        if not all([edge_id, edge_type, src_id, trg_id]):
            continue
            
        # Skip already processed edges and seg edges
        if edge_id in processed_edges or edge_type == 'seg':
            continue
            
        # Handle undercut relations
        if edge_type == 'und':
            if trg_id in edge_info:
                target_edge = edge_info[trg_id]
                target_src_id = target_edge['src']
                
                x_text = node_texts.get(src_id, "")
                y_text = node_texts.get(target_src_id, "")
                
                dataset.append({
                    'X': x_text,
                    'Y': y_text,
                    'Z': edge_type,
                    'topic': topic_id,
                    'Source': filename or 'unknown'
                })
        
        # Handle regular relations (sup, reb, etc.)
        elif edge_type not in ['add']:  # Skip add as we processed them above
            x_text = node_texts.get(src_id, "")
            y_text = node_texts.get(trg_id, "")
            
            dataset.append({
                'X': x_text,
                'Y': y_text,
                'Z': edge_type,
                'topic': topic_id,
                'Source': filename or 'unknown'
            })
    
    return dataset


def main():
    """Main function to run the XML to CSV conversion."""
    folder_path = input("Enter the folder path containing XML files: ").strip()
    
    if not os.path.isdir(folder_path):
        print("Invalid folder path")
        return
    
    separate = input("Save separate CSV for each XML file? (y/n): ").strip().lower()
    separate_files = separate == 'y'
    
    dataset = process_xml_files(folder_path, separate_files)
    
    if not dataset:
        print("No data extracted")
        return
    
    # Save combined dataset
    csv_filename = 'combined_arggraph_dataset.csv'
    with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['X', 'Y', 'Z','topic', 'Source']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(dataset)
    
    print(f"Extracted {len(dataset)} relations total")
    print(f"Combined dataset saved to {csv_filename}")
    
    # Display file statistics
    file_stats = {}
    for item in dataset:
        source = item['Source']
        file_stats[source] = file_stats.get(source, 0) + 1
    
    print("File statistics:")
    for filename, count in file_stats.items():
        print(f"  {filename}: {count} relations")

if __name__ == '__main__':
    main()

Enter the folder path containing XML files: corpus
Save separate CSV for each XML file? (y/n): y
Processing: corpus\micro_c001.xml
hunting_improves_environment
Saved to micro_c001_dataset.csv
Extracted 6 relations from micro_c001.xml
Processing: corpus\micro_c002.xml
hunting_improves_environment
Saved to micro_c002_dataset.csv
Extracted 3 relations from micro_c002.xml
Processing: corpus\micro_c003.xml
hunting_improves_environment
Saved to micro_c003_dataset.csv
Extracted 4 relations from micro_c003.xml
Processing: corpus\micro_c004.xml
hunting_improves_environment
Saved to micro_c004_dataset.csv
Extracted 3 relations from micro_c004.xml
Processing: corpus\micro_c005.xml
older_people_better_parents
Saved to micro_c005_dataset.csv
Extracted 6 relations from micro_c005.xml
Processing: corpus\micro_c006.xml
older_people_better_parents
Saved to micro_c006_dataset.csv
Extracted 5 relations from micro_c006.xml
Processing: corpus\micro_c007.xml
older_people_better_parents
Saved to micro_c007_d

In [19]:
len(sd1)

704

In [1]:
import pandas as pd


sd1 = pd.read_csv("combined_arggraph_dataset.csv")


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model_name = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
nli_tokenizer = AutoTokenizer.from_pretrained(model_name)
model_nli = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
def NLI(x,y,tokenizer,model):
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

    
    

    premise = x
    hypothesis = y

    input = tokenizer(premise, hypothesis, truncation=True, return_tensors="pt")
    output = model(input["input_ids"].to(device))  # device = "cuda:0" or "cpu"
    prediction = torch.softmax(output["logits"][0], -1).tolist()
    label_names = ["entailment", "neutral", "contradiction"]
    prediction = {name: round(float(pred) * 100, 1) for pred, name in zip(prediction, label_names)}
#     print(prediction)
    return max(prediction, key=prediction.get),max(prediction.values())

C:\Users\fxy19\anaconda3\envs\eesnli\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from tqdm import tqdm

In [9]:
neu = []
for _, g in tqdm(sd1.groupby(["Source"])):
    print(g)
    X_N = []
    Y_N = []
    check = []
    for i in g.iterrows():
        X_N.append(i[1]["X"])
        Y_N.append(i[1]["Y"])
        check.append([i[1]["X"],i[1]["Y"]])
        topic = i[1]['topic']
    for i in X_N:
        for j in Y_N:
            if i==j:
                continue
            if [i,j] in check or [j,i] in check:
                continue
            else:
                print(i,j)
                if [i,j,topic,'neutral'] not in neu:
                    cn = NLI(i,j,nli_tokenizer,model_nli)
                    if cn[0] == 'neutral' and cn[1]>99:
                        neu.append([i,j,topic,'neutral'])
#                 if [j,i] not in neu:
#                     cn = NLI(j,i,nli_tokenizer,model_nli)
#                     if cn[0] == 'neutral' and cn[1]>90:
#                         neu.append([j,i,topic,'neutral'])

  0%|          | 0/171 [00:00<?, ?it/s]

                                                   X  \
0  because overpopulated species can be thinned out.   
1  Getting rid of an overpopulation enables the s...   
2  It also allows for nature to take back the woo...   
3  Some people may object to hunting on the basis...   
4     Animals do not necessarily feel pain as we do,   
5  and if there are not enough resources to susta...   

                                                   Y    Z  \
0                Hunting is good for the environment  sup   
1  because overpopulated species can be thinned out.  sup   
2  because overpopulated species can be thinned out.  sup   
3                Hunting is good for the environment  reb   
4  Some people may object to hunting on the basis...  reb   
5  Some people may object to hunting on the basis...  reb   

                          topic          Source  
0  hunting_improves_environment  micro_c001.xml  
1  hunting_improves_environment  micro_c001.xml  
2  hunting_improves_environme

  1%|          | 1/171 [00:00<01:08,  2.48it/s]

It also allows for nature to take back the woods and grasslands, which will also enable other wildlife to grow. Some people may object to hunting on the basis of humane treatment of animals.
Some people may object to hunting on the basis of humane treatment of animals. because overpopulated species can be thinned out.
Some people may object to hunting on the basis of humane treatment of animals. because overpopulated species can be thinned out.
Animals do not necessarily feel pain as we do, Hunting is good for the environment
Animals do not necessarily feel pain as we do, because overpopulated species can be thinned out.
Animals do not necessarily feel pain as we do, because overpopulated species can be thinned out.
Animals do not necessarily feel pain as we do, Hunting is good for the environment
and if there are not enough resources to sustain an overpopulation, then a quick bullet or arrow is better than slow starvation. Hunting is good for the environment
and if there are not enoug

  2%|▏         | 3/171 [00:00<00:31,  5.31it/s]

...

because of not being around just adults all the time. A larger family will create a better atmosphere for a child.
In my experience, being an only child you are more serious. In the long run, yes a larger family is better for children,
In my experience, being an only child you are more serious. In the long run, yes a larger family is better for children,
In my experience, being an only child you are more serious. because they will learn so much more.
In my experience, being an only child you are more serious. They will be more outgoing, more playful,
                                                     X  \
684  Statistics show there is a larger ratio of tee...   
685  The main problem is the loss of parental focus...   
686  One possible counter argument is that there is...   
687  I would suggest that this is needed because of...   

                                                     Y    Z  \
684    Large families are not better for the children.  sup   
685    Large families are 

 99%|█████████▉| 170/171 [00:22<00:00,  9.36it/s]

Parents don't live forever, and it can be very comforting to know you will have siblings that can still form a family with you. You could argue that children in large families don't receive as much one-on-one attention from their parents.
Parents don't live forever, and it can be very comforting to know you will have siblings that can still form a family with you. You could argue that children in large families don't receive as much one-on-one attention from their parents.
IF you don't get along as well with one or more siblings, you don't feel isolated because there are undoubtedly other siblings you can interact with. You could argue that children in large families don't receive as much one-on-one attention from their parents.
IF you don't get along as well with one or more siblings, you don't feel isolated because there are undoubtedly other siblings you can interact with. You could argue that children in large families don't receive as much one-on-one attention from their parents.


100%|██████████| 171/171 [00:22<00:00,  7.59it/s]

However, this could also cause disturbances and jealousy. Large families are not necessarily better for children.
However, this could also cause disturbances and jealousy. Large families are not necessarily better for children.
Smaller families have the potential to be tighter and more close knit due to their size. One could argue that with a larger family, the children would have more people to rely on.


In [17]:
NLI("lady match","eat lunch",nli_tokenizer,model_nli)

('contradiction', 83.3)

In [10]:
len(neu)

400

In [12]:
pd.DataFrame(neu).to_csv("neu2.csv")

In [11]:
neu

[['because overpopulated species can be thinned out.',
  'Some people may object to hunting on the basis of humane treatment of animals.',
  'hunting_improves_environment',
  'neutral'],
 ['Getting rid of an overpopulation enables the smaller animals in the food chain to grow.',
  'Some people may object to hunting on the basis of humane treatment of animals.',
  'hunting_improves_environment',
  'neutral'],
 ['It also allows for nature to take back the woods and grasslands, which will also enable other wildlife to grow.',
  'Some people may object to hunting on the basis of humane treatment of animals.',
  'hunting_improves_environment',
  'neutral'],
 ['Animals do not necessarily feel pain as we do,',
  'because overpopulated species can be thinned out.',
  'hunting_improves_environment',
  'neutral'],
 ['In the absence of wolves and other large predators, humans become the deer population control to keep to deer from over-grazing regional flora.',
  'Hunting is good.',
  'hunting_im